# 01 · Training & Baseline Capture

Fits a `RandomForestRegressor` on the California Housing dataset, then serialises
one `kll_floats_sketch` per feature **and** for the prediction into compact
base64 strings stored in `baselines/baseline.json`.

Run this notebook **once** before any inference run.

In [ ]:
%pip install \
    scikit-learn==1.6.1 \
    datasketches==5.2.0 \
    scipy==1.15.3 \
    numpy==2.2.5 \
    pandas==2.2.3 \
    joblib==1.4.2 \
    --quiet

In [ ]:
import base64
import json
import pathlib

import joblib
import numpy as np
import pandas as pd
from datasketches import kll_floats_sketch
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [ ]:
BASELINES_DIR = pathlib.Path("baselines")
DATA_DIR = pathlib.Path("data")
BASELINES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

KLL_K = 200
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Fix global random state so every run produces identical outputs
import random

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 1 · Load data

In [ ]:
def load_california_housing() -> tuple[pd.DataFrame, pd.Series]:
    """Load California Housing as a DataFrame.

    Returns
    -------
    X : pd.DataFrame
        Feature matrix with named columns.
    y : pd.Series
        Median house value (100 k USD), named 'MedHouseVal'.

    Examples
    --------
    >>> X, y = load_california_housing()
    >>> X.shape
    (20640, 8)
    """
    dataset = fetch_california_housing(as_frame=True)
    return dataset.data, dataset.target


X, y = load_california_housing()
print(f"Dataset: {X.shape[0]:,} rows x {X.shape[1]} features")
X.head()

## 2 · Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_test.to_parquet(DATA_DIR / "X_test.parquet", index=False)
y_test.to_frame().to_parquet(DATA_DIR / "y_test.parquet", index=False)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

## 3 · Train model

In [ ]:
def train_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    random_state: int = RANDOM_STATE,
) -> RandomForestRegressor:
    """Train a RandomForestRegressor and return the fitted estimator.

    Parameters
    ----------
    X_train : pd.DataFrame
        Training features.
    y_train : pd.Series
        Training targets.
    random_state : int
        Reproducibility seed.

    Returns
    -------
    RandomForestRegressor
        Fitted model.

    Examples
    --------
    >>> model = train_model(X_train, y_train)
    >>> model.n_estimators
    100
    """
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        n_jobs=-1,
        random_state=random_state,
    )
    model.fit(X_train, y_train)
    return model


model = train_model(X_train, y_train)
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"R2 train={train_score:.4f}  test={test_score:.4f}")

joblib.dump(model, DATA_DIR / "model.joblib")
print("Model saved -> data/model.joblib")

## 4 · Build KLL sketches & serialise baselines

One sketch per training column + one for training predictions.
Serialised as `serialize()` -> bytes -> base64 string stored in JSON.

In [ ]:
def build_kll_sketch(values: np.ndarray, k: int = KLL_K) -> kll_floats_sketch:
    """Build a KLL float sketch from a 1-D array, skipping NaNs.

    Parameters
    ----------
    values : np.ndarray
        1-D float array.
    k : int
        Sketch accuracy parameter.

    Returns
    -------
    kll_floats_sketch
        Populated sketch.

    Examples
    --------
    >>> sk = build_kll_sketch(np.arange(1000, dtype=float))
    >>> sk.get_quantile(0.5)
    499.5
    """
    sketch = kll_floats_sketch(k)
    clean = values[~np.isnan(values)].astype(np.float32)
    for v in clean:
        sketch.update(v)
    return sketch


def sketch_to_base64(sketch: kll_floats_sketch) -> str:
    """Serialise a KLL sketch to a base64-encoded UTF-8 string.

    Parameters
    ----------
    sketch : kll_floats_sketch
        Populated sketch.

    Returns
    -------
    str
        Base64 string suitable for embedding in JSON.

    Examples
    --------
    >>> sk = kll_floats_sketch(200)
    >>> sk.update(1.0)
    >>> isinstance(sketch_to_base64(sk), str)
    True
    """
    return base64.b64encode(sketch.serialize()).decode("utf-8")


def base64_to_sketch(b64: str) -> kll_floats_sketch:
    """Deserialise a KLL sketch from a base64-encoded string.

    Parameters
    ----------
    b64 : str
        Base64 string produced by :func:`sketch_to_base64`.

    Returns
    -------
    kll_floats_sketch
        Deserialised sketch.

    Examples
    --------
    >>> sk = base64_to_sketch(sketch_to_base64(build_kll_sketch(np.arange(100, dtype=float))))
    >>> sk.n
    100
    """
    return kll_floats_sketch.deserialize(base64.b64decode(b64))


train_predictions = model.predict(X_train).astype(np.float32)

baseline: dict = {"k": KLL_K, "features": {}, "prediction": {}}

for col in X_train.columns:
    sk = build_kll_sketch(X_train[col].to_numpy())
    baseline["features"][col] = {
        "sketch_b64": sketch_to_base64(sk),
        "n": sk.n,
        "min": float(sk.get_min_value()),
        "max": float(sk.get_max_value()),
    }

pred_sk = build_kll_sketch(train_predictions)
baseline["prediction"] = {
    "sketch_b64": sketch_to_base64(pred_sk),
    "n": pred_sk.n,
    "min": float(pred_sk.get_min_value()),
    "max": float(pred_sk.get_max_value()),
}

baseline_path = BASELINES_DIR / "baseline.json"
baseline_path.write_text(json.dumps(baseline, indent=2))
print(f"Baseline saved -> {baseline_path}")
print(f"  Features : {list(baseline['features'].keys())}")
print(f"  File size: {baseline_path.stat().st_size / 1024:.1f} KB")

## 5 · Sanity check — decode a sketch and inspect quantiles

In [ ]:
loaded = json.loads(baseline_path.read_text())
medinc_sketch = base64_to_sketch(loaded["features"]["MedInc"]["sketch_b64"])

quantile_ranks = [0.1, 0.25, 0.5, 0.75, 0.9]
quantile_values = medinc_sketch.get_quantiles(quantile_ranks)

print("MedInc quantiles from decoded baseline sketch:")
for rank, value in zip(quantile_ranks, quantile_values):
    print(f"  p{int(rank * 100):02d} = {value:.4f}")

## 6 · Non-encoded baseline — human-readable structure

Same shape as `baseline.json` but the sketch is replaced by a plain dict of
representative quantiles so you can inspect the distribution at a glance.
**Not** suitable for re-loading into a KLL sketch — use `baseline.json` for that.

In [ ]:
def sketch_to_readable_summary(
    sketch: kll_floats_sketch,
    quantile_ranks: list[float] | None = None,
) -> dict:
    """Convert a KLL sketch into a human-readable quantile summary dict.

    Replaces the opaque base64 blob with labelled quantile values and basic
    descriptive statistics for visual inspection.

    Parameters
    ----------
    sketch : kll_floats_sketch
        Populated sketch.
    quantile_ranks : list[float] | None
        Ranks to evaluate. Defaults to [0.01, 0.05, 0.10, 0.25, 0.50,
        0.75, 0.90, 0.95, 0.99].

    Returns
    -------
    dict
        Keys: ``n``, ``min``, ``max``, ``quantiles`` (rank -> value).

    Examples
    --------
    >>> sk = kll_floats_sketch(200)
    >>> for v in range(100): sk.update(float(v))
    >>> summary = sketch_to_readable_summary(sk)
    >>> list(summary.keys())
    ['n', 'min', 'max', 'quantiles']
    """
    if quantile_ranks is None:
        quantile_ranks = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

    values = sketch.get_quantiles(quantile_ranks)
    return {
        "n": sketch.n,
        "min": float(sketch.get_min_value()),
        "max": float(sketch.get_max_value()),
        "quantiles": {
            f"p{int(r * 100):02d}": float(v)
            for r, v in zip(quantile_ranks, values)
        },
    }


non_encoded: dict = {"k": KLL_K, "features": {}, "prediction": {}}

for col in X_train.columns:
    sk = build_kll_sketch(X_train[col].to_numpy())
    non_encoded["features"][col] = sketch_to_readable_summary(sk)

pred_sk = build_kll_sketch(model.predict(X_train).astype("float32"))
non_encoded["prediction"] = sketch_to_readable_summary(pred_sk)

non_encoded_path = BASELINES_DIR / "non_encoded_baseline.json"
non_encoded_path.write_text(json.dumps(non_encoded, indent=2))
print(f"Non-encoded baseline saved -> {non_encoded_path}")
print(f"  File size: {non_encoded_path.stat().st_size / 1024:.1f} KB")
print()
print("MedInc summary:")
print(json.dumps(non_encoded["features"]["MedInc"], indent=2))